<H1>Compute variables from a formatted dataset<H1>

<H2> Import packages <H2>

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

<H2> Read and check the dataset <H2> 

In [ ]:
data = pd.read_csv('/Path/to/Combined_trial_data.csv')

In [ ]:
data.head()

In [ ]:
#Uncomment if there is an extra column at the beginning
#data = data.drop(data.columns[[0]], axis = 1)
#data.head()

In [ ]:
data['response'].isna().sum() #The response variable doesn't have any missing values

In [ ]:
data['rt'].isna().sum() #The reaction time also doesn't have any missing values

In [ ]:
#Option to omit reaction time values > 10000 (amounting to 10 sec) in the dataset
data['rt'] = np.where(data['rt'] > 10000, np.nan, data['rt'])

<H2> Set function parameters <H2>

In [ ]:
grouping_variables = ['id', 'set_size']
grouping_variables_string = "id_and_set_size" #Specify the variables used for grouping the dataset
group_by_set_size = True
filter_set_sizes = False #If True, input which set sizes to omit below
set_sizes_to_omit = [6]
filter_rows = True #If True, input the max rows per participant within each set size. Row numbers larger than this will be omitted.
max_row_number = 60
omit_rt_outliers = False #If true, extreme rt values will be omitted from the dataset before computing the reaction time summaries

<H2> Omit outlying reaction time values<H2>
    (if specified above) 

In [ ]:
def remove_group_outliers(df, group_cols, value_col, iqr_multiplier=1.5):
    
    """
    Removes outliers from a dataframe based on group-specific IQR calculations.
    
    Parameters:
    - df: Pandas DataFrame
    - group_cols: List of column names to group by
    - value_col: Name of the numerical column to check for outliers
    - iqr_multiplier: Multiplier for IQR to define bounds (default 1.5)
    
    Returns:
    - Filtered dataframe with outliers omitted
    
    """
    
    # Calculate Q1, Q3, and IQR for each group
    q1 = df.groupby(group_cols)[value_col].transform('quantile', 0.25)
    q3 = df.groupby(group_cols)[value_col].transform('quantile', 0.75)
    iqr = q3 - q1
    
    # Define bounds
    lower_bound = q1 - (iqr_multiplier * iqr)
    upper_bound = q3 + (iqr_multiplier * iqr)
    
    # Set values that fall outside the bounds to na
    df[value_col] = np.where(df[value_col] < lower_bound, np.nan, df[value_col])
    df[value_col] = np.where(df[value_col] > upper_bound, np.nan, df[value_col])
    
    df = df.reset_index()
    
    return df

In [ ]:
#Run the function on the data
#Call the function on the reaction time column, grouped by id
if omit_rt_outliers == True :
    data = remove_group_outliers(data, 'id', 'rt', iqr_multiplier=1.5)
else : 
    print("All reaction time values were included.")

In [ ]:
data.shape

<H2>Trim the dataset<H2> (if/as specified above) 

The final dataset will contain the desired set sizes and number of trials per participant and set size.

In [ ]:
def trim_dataset(df, filter_set_sizes, filter_rows):

    """
    Removes rows from a dataframe above the max row number specified.
    Removes specified set sizes from the dataframe
    
    Parameters:
    - df: Pandas DataFrame
    - trim_set_sizes: True or False. If False, the dataframe remains unaltered. 
    - trim_rows: True or False. If False, the dataframe remains unaltered. 
    
    Returns:
    - Filtered dataframe with specified rows and set sizes omitted. 
    """

    global filtered_data

    if filter_rows == True : 
        n = max_row_number # Keep the first n rows within each id and set size
        df = df.groupby(['id', 'set_size']).head(n)
        print('Row numbers >', max_row_number, 'within each set size were omitted.')

    else :
        print("All rows were included.")

    if filter_set_sizes == True : 
        for set_size in set_sizes_to_omit :
            df = df[df['set_size'] != set_size]
            print('Set sizes', set_sizes_to_omit, 'were omitted.')
    else  :
        print("All set sizes were included.")

    filtered_data = df.reset_index()   
    
    return filtered_data

In [ ]:
trim_dataset(data, filter_set_sizes, filter_rows)

In [ ]:
filtered_data.shape #Confirm that the dimensions are as expected

<H2>Numeric variable (reaction time) </H2>

Compute the trial counts, mean, standard deviation, and quantile values of a numeric variable (rt).

In [ ]:
def numeric_variable_stats(df, group_cols, value_col):
    
    """
    Parameters:
    - df: Pandas DataFrame
    - group_cols: the levels of the dataframe within which the variables will be computed. 
    - value_col: Expected to be reaction time.  
    
    Returns:
    - a dataframe summarizing the mean, standard deviation, and quantile values
    (0, .25, .50, .75, 1) within each level of a numeric variable
    within the original grouped DataFrame. 
    
    """
    
    global numeric_output
    
    df = df.groupby(group_cols)[value_col].describe(percentiles=[0.25, 0.5, 0.75]).reset_index()
    
    #Alternative method to get just the mean and SD
    #df.groupby(group_cols)[value_col].agg(mean="mean", std="std").reset_index()    

    if group_by_set_size == True :
        
        colnames = ['id', 'set_size', 'trial_count', 'rt_mean', 'rt_sd', 'rt_min', 'rt_q1',
                'rt_median', 'rt_q3', 'rt_max']
    else :
        colnames = ['id', 'trial_count', 'rt_mean', 'rt_sd', 'rt_min', 'rt_q1',
                'rt_median', 'rt_q3', 'rt_max']
        
    df = pd.DataFrame(df)
    df = df.set_axis(colnames, axis=1)

    numeric_output = df
    
    return numeric_output    

In [ ]:
#Call the function on the dataset
numeric_variable_stats(filtered_data, grouping_variables, "rt")

<H2> Categorical variables <H2>

Summarize the trial counts of each condition (combining change and response).

In [ ]:
def categorical_variable_counts(df, group_cols, value_col):

    """
    Counts up the total number of hits, false alarms, correction rejections, and misses
    within each level of a grouped dataframe
    0 = same color
    1 = different color
    hits are trials where change = 1 and response = 1 
    false alarms are trials where change = 0 and response = 1 
    correct rejections are trials where change = 0 and response = 0
    misses are trials where change = 1 and response = 0
    
    Parameters:
    - df: Pandas DataFrame
    - group_cols: the levels of the dataframe within which the variables will be computed. 
    - value_col: Any will do, since the function only counts up the number of rows in each condition. 

    Returns:
    - a dataframe providing the total counts of hits, false alarms, correct rejections,
    and misses within the original grouped dataframe.
    
    """
    
    global counts

    permanent_cols = ['change', 'response']
    final_group_cols =  list([*group_cols, *permanent_cols])

    df = df.groupby(final_group_cols)[value_col].size().unstack(fill_value=0).unstack(fill_value=0)
    df = pd.DataFrame(df)
    df.columns = ['correct_rejections', 'misses', 'false_alarms', 'hits']

    counts = df.reset_index()

    return counts

In [ ]:
#Call the function on the dataset
categorical_variable_counts(filtered_data, grouping_variables, "trial_num")

<H2> Create final analysis variables </H2>

In [ ]:
def create_analysis_variables(df):

    """
    Creates the analysis variables
    accuracy is the # correct responses / # responses
    d' is the sensitivity to a true change, computed as z(hit rate) - z(false alarm rate)
    A' is a non-parametric measure of sensitivity. 
    K is a measure of capacity that approximates how many items are simultaneously 
    held in working memory. K can only be computed if the dataframe was grouped by set size. 
    Otherwise, an empty column is returned. 
    
    Parameters:
    - df: Pandas DataFrame expected to be the counts dataframe from the previous function
    
    Returns:
    - a dataframe providing the final variables for analysis, including
    the accuracy, d', A', response bias and K.  
    
    """
    
    global analysis_variables
    
    #Compute rate variables with a correction for extreme values
    df['accuracy'] = (df['hits'] + df['correct_rejections'] + .5)/(df['hits'] + df['correct_rejections'] + df['false_alarms'] + df['misses'] + 1)
    df['hit_rate'] = (df['hits'] + .5) /(df['hits'] + df['misses'] + 1)
    df['false_alarm_rate'] = (df['false_alarms'] + .5) / (df['correct_rejections'] + df['false_alarms'] + 1)
    df['correct_rejection_rate'] = (df['correct_rejections'] + .5) / (df['correct_rejections'] + df['false_alarms'] + 1)
    df['miss_rate'] = (df['misses'] + .5) / (df['hits'] + df['misses'] + 1)


    #Compute analysis variables

    #Estimation of d' (sensitivity to a true change)
    df['dprime'] = norm.ppf(df['hit_rate']) - norm.ppf(df['false_alarm_rate'])

    #A' is a non-parametric alternative to d'.
    df['aprime'] = np.where(df['hit_rate'] >= df['false_alarm_rate'],
                            (.5 + (((df['hit_rate'] - df['false_alarm_rate']) * (1 + df['hit_rate'] - df['false_alarm_rate'])) / (4 * df['hit_rate'] * (1 -  df['false_alarm_rate'])))),
                            (.5 - (((df['false_alarm_rate'] - df['hit_rate']) * (1 + df['false_alarm_rate'] - df['hit_rate'])) / (4 * df['false_alarm_rate'] * (1 -  df['hit_rate'])))))
    
    #Response bias to choose 'different' with a correction for extreme values
    df['response_bias'] = (df['hits'] + df['false_alarms'] + .5)/(df['hits'] + df['correct_rejections'] + df['false_alarms'] + df['misses'] + 1)
    df['response_bias_probability'] = (norm.ppf(df['hit_rate']) + norm.ppf(df['false_alarm_rate']))/2
    
    #Paschler's K, a measure of capacity that should remain stable across set sizes
    #This function has been modified such that values < 1 are diminishing positive fractions. 
    if group_by_set_size == True :
        df['k'] = df['set_size'] * (df['hit_rate'] - df['false_alarm_rate']) / (1 - df['false_alarm_rate'])
        df['k_modified'] = np.where(df['k'] < 1, 
                           (1/(1+np.exp(-df['k']))),
                           (df['k']))
    else :
        df['k'] = ''

    analysis_variables = df

    return analysis_variables

In [ ]:
#Run the function on the categorical dataset to create the new variables
create_analysis_variables(counts)

<H2> Combine and save the full dataset </H2>

In [ ]:
#Bind the categorical variable results to the rt variables
final_analysis_variables = pd.concat([numeric_output, analysis_variables], axis=1)
final_analysis_variables = final_analysis_variables.loc[:, ~final_analysis_variables.columns.duplicated()]

In [ ]:
print(final_analysis_variables.columns)

In [ ]:
#Add back the study and experiment columns
final_analysis_variables['study'] = np.where((final_analysis_variables['id'] < 215), 1, 2)
final_analysis_variables['experiment'] = np.where((final_analysis_variables['id'] <= 135) | (final_analysis_variables['id'] >= 215), 1, 2)

if group_by_set_size == True :
    final_analysis_variables = final_analysis_variables[['study', 'experiment', 'id', 'set_size', 'trial_count', 'rt_mean', 'rt_sd', 'rt_min', 'rt_q1', 'rt_median', 'rt_q3', 'rt_max', 'hits', 'false_alarms', 
           'correct_rejections', 'misses', 'accuracy', 'hit_rate', 'false_alarm_rate', 'correct_rejection_rate', 'miss_rate',
           'dprime', 'aprime', 'response_bias', 'response_bias_probability', 'k', 'k_modified']]
else : final_analysis_variables = final_analysis_variables[['study', 'experiment', 'id', 'trial_count', 'rt_mean', 'rt_sd', 'rt_min', 'rt_q1', 'rt_median', 'rt_q3', 'rt_max', 'hits', 'false_alarms', 
           'correct_rejections', 'misses', 'accuracy', 'hit_rate', 'false_alarm_rate', 'correct_rejection_rate', 'miss_rate',
           'dprime', 'aprime', 'response_bias', 'response_bias_probability', 'k', 'k_modified']]

In [ ]:
final_analysis_variables.head()

In [ ]:
#Write the dataset to a csv file with the grouping variables in the title. 
print('analysis variables were grouped by', grouping_variables)
print('filtering by set size was set to', filter_set_sizes, 'and trimming rows was set to', filter_rows)
print('set sizes', set_sizes_to_omit, 'were listed in set_sizes_to_omit and row number', max_row_number, 'was listed as the max_row_number within each set size.')
print('The dataset with trials has dimensions:', filtered_data.shape)
print('The final dataset with variables has dimensions:', final_analysis_variables.shape)
filename = 'analysis_variables_grouped_by_' + grouping_variables_string + '.csv'

#Write the csv file with the analysis variables
final_analysis_variables.to_csv(filename, index=False)

#Option to write the trimmed trial-level dataset to a csv file:
filtered_data.to_csv("filtered_trial_data.csv", index=False)